In [6]:
from func_steinhaurer import *
from _functions4plasma import *

import math
import numpy as np 
import sympy as sp 
from scipy.optimize import root
import matplotlib.pyplot as plt 

# Define Variables
E0, E1, E2, E3 = sp.symbols("E:4")
B, r, z, a, b, X = sp.symbols('B r z a b X')
psi = sp.symbols('P')

### === External Psi
# Define Equations (for Eq. 2)
T1 = E0*(r**2/a**2)
T2 = E1*(r**2/a**2)*(r**2/a**2 - 4*z**2/a**2)
T3 = E2*(((E3*b + z)/((r**2 + (E3*b+z)**2)**0.5))+((E3*b - z)/((r**2 + (E3*b-z)**2)**0.5)))

psi         = sp.simplify((B*a**2)/2*(T1+T2+T3))
psi_r       = sp.diff(psi, r)
psi_rr      = sp.diff(psi_r, r)
psi_z       = sp.diff(psi, z)
psi_zz      = sp.diff(psi_z, z)
psi_rz      = sp.diff(psi_r, z)

# Define Functions
f_psi       = sp.lambdify((B, E0, E1, E2, E3, r, z, a, b), psi)
f_psi_r     = sp.lambdify((B, E0, E1, E2, E3, r, z, a, b), psi_r)
f_psi_rr    = sp.lambdify((B, E0, E1, E2, E3, r, z, a, b), psi_rr)
f_psi_z     = sp.lambdify((B, E0, E1, E2, E3, r, z, a, b), psi_z)
f_psi_zz    = sp.lambdify((B, E0, E1, E2, E3, r, z, a, b), psi_zz)
f_psi_rz    = sp.lambdify((B, E0, E1, E2, E3, r, z, a, b), psi_rz)


# Curvature based of Steinhaurer's analytical equation (Eq. 2)
def ext_curvature(B, E0, E1, E2, E3, r, z, a, b):
    num = f_psi_rr(B, E0, E1, E2, E3, r, z, a, b)*(f_psi_z(B, E0, E1, E2, E3, r, z, a, b))**2 - 2*f_psi_rz(B, E0, E1, E2, E3, r, z, a, b)*f_psi_r(B, E0, E1, E2, E3, r, z, a, b)*f_psi_z(B, E0, E1, E2, E3, r, z, a, b) + f_psi_zz(B, E0, E1, E2, E3, r, z, a, b)*(f_psi_r(B, E0, E1, E2, E3, r, z, a, b))**2
    den = ((f_psi_r(B, E0, E1, E2, E3, r, z, a, b))**2 + (f_psi_z(B, E0, E1, E2, E3, r, z, a, b))**2)**(1.5)
    return np.abs(num/den)

### === Internal Psi
i_psi       = ((3/2)**0.5)*((X*B*r**2)/2)*(1-(r**2/a**2)-(z**4/b**4)+1.5*(a**2/b**2))
i_psi_r     = sp.diff(i_psi, r)
i_psi_rr    = sp.diff(i_psi_r, r)
i_psi_z     = sp.diff(i_psi, z)
i_psi_zz    = sp.diff(i_psi_z, z)
i_psi_rz    = sp.diff(i_psi_r, z)

# Define Functions
fi_psi     = sp.lambdify((B, X, r, z, a, b), i_psi)
fi_psi_r   = sp.lambdify((B, X, r, z, a, b), i_psi_r)
fi_psi_rr  = sp.lambdify((B, X, r, z, a, b), i_psi_rr)
fi_psi_z   = sp.lambdify((B, X, r, z, a, b), i_psi_z)
fi_psi_zz  = sp.lambdify((B, X, r, z, a, b), i_psi_zz)
fi_psi_rz  = sp.lambdify((B, X, r, z, a, b), i_psi_rz)

# Curvature based of Steinhaurer's analytical equation (Eq. 2)
def int_curvature(B, X, r, z, a, b):
    num = fi_psi_rr(B, X, r, z, a, b)*(fi_psi_z(B, X, r, z, a, b))**2 - 2*fi_psi_rz(B, X, r, z, a, b)*fi_psi_r(B, X, r, z, a, b)*fi_psi_z(B, X, r, z, a, b) + fi_psi_zz(B, X, r, z, a, b)*(fi_psi_r(B, X, r, z, a, b))**2
    den = ((fi_psi_r(B, X, r, z, a, b))**2 + (fi_psi_z(B, X, r, z, a, b))**2)**(1.5)
    return np.abs(num/den)

Lconv = 1e3
acc = 100
# Parameters / Domain Definitions 
Rw          = 6.1/Lconv                 # Wall Radius [m]
Rc          = Rw
zLen        = 200/Lconv                 # Liner length [m]

B0  = 30
sig = f = 1.5                           # Flare parameter; adjustable parameter

T           = np.array([50])            # Temp in [eV]
TK          = T * eV/kb                 # Temp number in [K]

Rmax        = Rw                        # maximum r-domain value [m]
Rmin        = 0                         # minimum r-domain value [m]
Nr          = 1000                      # number of points in r-direction [\]
dr          = (Rmax - Rmin) / Nr        # radial differential [m]
Zmax        = zLen / 2                  # maximum z-domain value [m]
Zmin        = -zLen / 2                 # minimum z-domain value [m]
Nz          = 1000                      # number of points in z-direction [\]
dz          = (Zmax - Zmin) / Nz        # axial differential [m]
domx        = 1.4                       # domain multiplier to make plotting modifications easier
domy        = 0.8                       # domain multiplier to make plotting modifications easier
Rd          = Rmax * domx               # domain radius [m]
Zd          = Zmax * domy               # domain z-length [m]
r = np.arange(-Rd, Rd+dr, dr)
z = np.arange(-Zd, Zd+dz, dz)    
r_mesh, z_mesh = np.meshgrid(r, z, indexing='ij')


conI = True             # Varying Xs and Varying Elongation
conII = False           # Constant Xs and Varying Elongation
conIII = False          # Varying Xs and Constant Elongation

if conI:
    elong_arr   = np.linspace(1,10,num = acc)
    Xs_arr      = np.linspace(0.3, 0.9,num = acc)
    a_arr       = Rc*Xs_arr
    b_arr       =  np.multiply(a_arr, elong_arr)
    Bw_arr      = B0 / (1-(Xs_arr)**2)
elif conII:
    elong_arr   = np.linspace(1,10,num = acc)
    Xs          = 0.6
    a           = Rc*Xs
    b_arr       =  a*elong_arr
    Bw          = B0 / (1 - Xs**2)
elif conIII: 
    E           = 4.5
    Xs_arr      = np.linspace(0.3, 0.9,num = acc)
    a_arr       = Rc*Xs_arr
    b_arr       = a_arr*E
    Bw_arr = B0 / (1-(Xs_arr)**2)
else:
    sys.exit()


def get_flux_contours(psi, R, Z, psi_level_mag, return_all=False):
    # returns rz 
    # 1) Build a tiny OFF‐SCREEN figure, extract contours, then close it:
    plt.ioff()                     # turn off interactive showing
    fig, ax = plt.subplots(figsize=(0.1,0.1))  
    cs = ax.contour(R, Z, psi, levels=[psi_level_mag])
    plt.close(fig)                 # immediately close it so nothing pops up

    # 2) Now extract the contour segments from cs:
    try:
        idx = list(cs.levels).index(psi_level_mag)
    except ValueError:
        raise RuntimeError(f"No ψ={psi_level_mag} level found in cs.levels={cs.levels}")
    segs = cs.allsegs[idx]
    if not segs:
        raise RuntimeError(f"No ψ={psi_level_mag} contour found in the domain.")

    # 3) Convert each Nx2 array into (r_i, z_i) loops exactly as you had:
    loops = []
    for seg in segs:
        verts = np.asarray(seg)    # shape=(Npts,2)
        r_i = verts[:,0].copy()
        z_i = verts[:,1].copy()
        loops.append((r_i, z_i))

    if return_all:
        return loops

    longest = max(loops, key=lambda pair: pair[0].shape[0])
    return longest


def psi_stein_N(r, z, a, b, N): 
    '''
    Tilt stability of a gyroviscous field-reversed configuration
    with realistic equilibria paper definition
    
    r: 2D np.array
    z: 2D np.array
    a: float; separatrix radius
    b: float; separatrix half-length
    N: float; shape index
    '''
    r_ = r/a
    z_ = z/a

    E = b/a
    D0 = (8*E**4 - 1) / (8*E**4 + 4*E**2)
    D1 = (4*E**2 + 1) / (8*E**4 + 4*E**2)
    B1 = (1-N) / (4*E**2*(1+2*D1) + N*(1+D1))

    g = 1 - r_**2 - (z_ / E)**2 + B1*(1-D0*(r_**2 - 4*z_**2) - D1*(r_**4 - 12*r_**2*z**2 + 8*z_**4))
    
    return 0.5*r**2*g

def difference(p1, p2): 
    return np.abs(p1-p2)

In [ ]:
e_params    = []
mask = list(range(0, acc))

rz_ext              = []
rz_int              = []
Br_ext_arr          = []
Br_int_arr          = []
Bz_ext_arr          = []
Bz_int_arr          = []

for i, E in enumerate(elong_arr):
        a   = a_arr[i]
        b   = b_arr[i]
        Xs  = Xs_arr[i]
        Bw  = Bw_arr[i]
        Br_int_arr.append((1/r_mesh) * internal_dpsi__dz_sporer(r_mesh, z_mesh, a, b, Bw, Xs))
        Bz_int_arr.append(-(1/r_mesh) * internal_dpsi__dr_sporer(r_mesh, z_mesh, a, b, Bw, Xs, f))

        initial_guess       = [Bw, Bw/3, Bw/5, 0.9]
        result              = root(external_E_params, initial_guess, args= (1/E, Xs, sig))

        if result.success: 
            E0, E1, E2, E3  = result.x
            e_params.append([E0, E1, E2, E3])
            psi_int = (internal_psi_sporer(r_mesh, z_mesh, a, b, Bw, Xs, f))
            psi_ext = (external_psi(r_mesh, z_mesh, Bw, a, b, E0, E1, E2, E3))

            rz_int.append(get_flux_contours(psi_int, r_mesh, z_mesh, 0, return_all=False))
            rz_ext.append(get_flux_contours(psi_ext, r_mesh, z_mesh, 0, return_all=False))

            dpsi__dr_ext    = external_dpsi__dr(r_mesh, z_mesh, Bw, a, b, E0, E1, E2, E3)       # [T*m^2]; gradient of
            dpsi__dz_ext    = external_dpsi__dz(r_mesh, z_mesh, Bw, a, b, E0, E1, E2, E3)       # external magnetic flux
        
            Br_ext_arr.append(-(1/r_mesh) * dpsi__dz_ext)            # radial magnetic field [T]          #[T*m]
            Bz_ext_arr.append((1/z_mesh) * dpsi__dr_ext)             # axial magnetic field [T]
        else: 
            mask.remove(i)


a_arr = a_arr[mask]
b_arr = b_arr[mask]
Xs_arr = Xs_arr[mask]
Bw_arr = Bw_arr[mask]
elong_arr = elong_arr[mask]

int_diff = []           # diff between psi_int and psi_int reconstructed
ext_diff = []           # diff between psi_ext and psi_ext reconstructed 
N_diff  = []            # diff between psi_int reconstructed and psi_ext reconstructed
for i, E in enumerate(elong_arr):
    a   = a_arr[i]
    b   = b_arr[i]
    Xs  = Xs_arr[i]
    Bw  = Bw_arr[i]
    e0, e1, e2, e3 = e_params[i]

    K_ext = (ext_curvature(Bw, e0, e1, e2, e3, a, 0, a, b))
    if math.isnan(K_ext):
        pass
    else:
        x1, y1 = [-5, 5], [b, b]
        N_ext = shape_index(a, b, K_ext)
        N_int = shape_index(a, b, int_curvature(Bw, Xs, a, 0, a, b))

        psi_int = psi_stein_N(r_mesh, z_mesh, a, b, N_int)
        psi_ext = psi_stein_N(r_mesh, z_mesh, a, b, N_ext)

        N_rz_int = get_flux_contours(psi_int, r_mesh, z_mesh, 0, return_all=False)
        N_rz_ext = get_flux_contours(psi_ext, r_mesh, z_mesh, 0, return_all=False)
        Npsi_int_peak  = np.interp(a, N_rz_int[0], N_rz_int[1])
        Npsi_ext_peak = np.interp(a, N_rz_ext[0], N_rz_ext[1])

        psi_int_peak = np.interp(a, rz_int[i][0], rz_int[i][1])
        psi_ext_peak = np.interp(a, rz_ext[i][0], rz_ext[i][1])
        
        int_diff.append(difference(psi_int_peak, Npsi_int_peak))
        ext_diff.append(difference(psi_ext_peak, Npsi_ext_peak))
        N_diff.append(difference(Npsi_int_peak, Npsi_ext_peak))

        ### === Plotting
        # fig, ax = plt.subplots(dpi=500)
        # ax.plot(rz_int[i][0], rz_int[i][1], label = r"$\psi_{int}$")
        # ax.plot(rz_ext[i][0], rz_ext[i][1], label = r"$\psi_{ext}$", linestyle=":")

        # ax.plot(N_rz_int[0], N_rz_int[1], label = r"reconstructed $\psi_{int}$" + f" N = {N_int:.3f}", linestyle=":")
        # ax.plot(N_rz_ext[0], N_rz_ext[1], label = r"reconstructed $\psi_{ext}$" + f" N = {N_ext:.3f}", linestyle="-.")
        
        # ax.set_xlim(-a*1.1, a*1.1)
        # ax.set_ylim(-b*1.1, b*1.1)
        # ax.plot(x1, y1, label = "z=b line")
        # ax.set_aspect('equal')
        # ax.legend(loc=8)
        # fig.savefig(f"contour{i}")



In [8]:
fig, ax = plt.subplots(dpi=500)
ax.plot(elong_arr, int_diff, label = r"$|\psi_{int} - \psi_{int, N}|$", c = 'purple')
ax.plot(elong_arr, ext_diff, label = r"$|\psi_{ext} - \psi_{ext, N}|$", c = 'orange')
ax.plot(elong_arr, N_diff, label = r"$|\psi_{ext, N} - \psi_{int, N}|$", c = 'blue')
ax.set_xlabel("Elongation")
ax.set_ylabel("Difference in y_values [m]")
ax.set_title("Difference in y at r=a")
ax.legend()
fig.savefig("differences")